# Vehicle Insurance Policy Lapse Prediction

## Model Training & Evaluation

This notebook focuses on data preprocessing, feature engineering, model training, threshold optimization, and performance evaluation for vehicle insurance policy lapse prediction.

Multiple machine learning models, preprocessing strategies, and imbalance handling approaches are evaluated to identify an effective baseline model for predicting policy lapse behavior in an imbalanced classification setting.

The following models were selected to compare different learning approaches:

- `Logistic Regression` - interpretable linear baseline model for imbalanced classification
- `Random Forest` - ensemble tree based model for capturing nonlinear relationships
- `XGBoost` - gradient boosting model optimized for predictive performance
- `CatBoost` - boosting algorithm designed for efficient handling of categorical features

In [1]:
# Standard Library
from pathlib import Path
import joblib
import os

# Data Manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

np.seterr(divide="ignore", invalid="ignore")

{'divide': 'warn', 'over': 'warn', 'under': 'ignore', 'invalid': 'warn'}

In [3]:
# Machine Learning - Preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

# Experiment Tracking
import mlflow

# Machine Learning - Model Selection
from sklearn.model_selection import train_test_split

# Machine Learning - Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

In [4]:
# Machine Learning - Evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Display Settings
pd.set_option("display.max_columns", None)

# Visualization Settings
sns.set_style("whitegrid")

### Load Data

Load the dataset and create a working copy for preprocessing while preserving the original data.

In [5]:
base_dir = Path().resolve().parent

data_path = base_dir / "data" / "eudirectlapse.csv"

if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found at: {data_path}")

df = pd.read_csv(data_path)

df.head()

,lapse,polholder_age,polholder_BMCevol,polholder_diffdriver,polholder_gender,polholder_job,policy_age,policy_caruse,policy_nbcontract,prem_final,prem_freqperyear,prem_last,prem_market,prem_pure,vehicl_age,vehicl_agepurchase,vehicl_garage,vehicl_powerkw,vehicl_region
0,0,38,stable,only partner,Male,normal,1,private or freelance work,1,232.46,4 per year,232.47,221.56,243.59,9,8,private garage,225 kW,Reg7
1,1,35,stable,same,Male,normal,1,private or freelance work,1,208.53,4 per year,208.54,247.56,208.54,15,7,private garage,100 kW,Reg4
2,1,29,stable,same,Male,normal,0,private or freelance work,1,277.34,1 per year,277.35,293.32,277.35,14,6,underground garage,100 kW,Reg7
3,0,33,down,same,Female,medical,2,private or freelance work,1,239.51,4 per year,244.40,310.91,219.95,17,10,street,75 kW,Reg5
4,0,50,stable,same,Male,normal,8,unknown,1,554.54,4 per year,554.55,365.46,519.50,16,8,street,75 kW,Reg14


### Train-Test Split

The dataset is split into training and testing sets before preprocessing to avoid data leakage.

Stratified sampling is applied to maintain the original class distribution of the target variable across both sets.

In [6]:
X = df.drop(columns="lapse")
y = df["lapse"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

Training set shape: (18448, 18)
Test set shape: (4612, 18)


#### Observations

- Stratified sampling preserved the original target class distribution across the training and testing sets.
- 80% of the data was used for training and 20% for testing.
- Preprocessing transformations will be fitted only on the training data and later applied to the test data to avoid data leakage.

## Data Preprocessing

A baseline preprocessing pipeline is created before model training. Numerical features are scaled, while categorical variables are encoded using one-hot encoding.

At this stage, the original feature set is retained without extensive feature engineering or manual category grouping in order to establish a clear baseline model performance.

In [7]:
numerical_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)

Numerical Features:
['polholder_age', 'policy_age', 'policy_nbcontract', 'prem_final', 'prem_last', 'prem_market', 'prem_pure', 'vehicl_age', 'vehicl_agepurchase']

Categorical Features:
['polholder_BMCevol', 'polholder_diffdriver', 'polholder_gender', 'polholder_job', 'policy_caruse', 'prem_freqperyear', 'vehicl_garage', 'vehicl_powerkw', 'vehicl_region']


In [8]:
baseline_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numerical",
            StandardScaler(),
            numerical_features
        )
    ]
)

In [9]:
def evaluate_model(model, X_test, y_test):

    y_pred = model.predict(X_test)

    y_prob = model.predict_proba(X_test)[:, 1]

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1_score": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_prob)
    }

    for metric_name, metric_value in metrics.items():
        print(f"{metric_name}: {metric_value:.4f}")

        mlflow.log_metric(metric_name, metric_value)

    return pd.DataFrame([metrics])

In [10]:
def train_and_evaluate_model(
    model,
    model_name,
    preprocessor,
    X_train,
    y_train,
    X_test,
    y_test,
    params=None
):

    with mlflow.start_run(run_name=model_name):

        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model)
        ])

        pipeline.fit(X_train, y_train)

        metrics = evaluate_model(
            pipeline,
            X_test,
            y_test
        )

        mlflow.log_param("model", model.__class__.__name__)

        if params:
            mlflow.log_params(params)

        mlflow.sklearn.log_model(
            pipeline,
            name=f"{model_name}_pipeline"
        )

    return pipeline, metrics

## Baseline Logistic Regression

Logistic Regression is used as the initial baseline model for policy lapse prediction. The model provides a simple and interpretable starting point for evaluating classification performance on the imbalanced dataset.

Class imbalance handling is enabled using `class_weight="balanced"` to improve the model’s ability to identify lapse cases.

The baseline model combines preprocessing and classification into a single pipeline to ensure consistent transformations and avoid data leakage during training and evaluation.

In [11]:
logistic_regression = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

logreg_pipeline, logreg_metrics = train_and_evaluate_model(
    model=logistic_regression,
    model_name="logistic_regression_baseline",
    preprocessor=baseline_preprocessor,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    params={
        "class_weight": "balanced",
        "max_iter": 1000
    }
)

logreg_metrics

2026/05/21 12:58:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


accuracy: 0.5917
precision: 0.1721
recall: 0.5736
f1_score: 0.2647
roc_auc: 0.6068


,accuracy,precision,recall,f1_score,roc_auc
0,0.591717,0.172081,0.573604,0.26474,0.606801


#### Observations

- The balanced Logistic Regression model achieved substantially higher recall, improving identification of lapse cases.
- The increase in recall came at the cost of lower precision and overall accuracy due to more false positive predictions.
- ROC-AUC indicates moderate discriminatory performance on the imbalanced dataset.
- The model provides a reasonable baseline for further model comparison and optimization.

## Random Forest Classifier

Random Forest is evaluated as a tree based ensemble model capable of capturing nonlinear relationships and feature interactions within the dataset.

Unlike Logistic Regression, Random Forest does not assume linear relationships between features and the target variable, making it suitable for handling complex behavioral patterns associated with policy lapse prediction.

The model is trained using the same preprocessing pipeline to ensure consistent data handling and fair comparison with the baseline Logistic Regression model.

In [12]:
random_forest = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

random_forest_pipeline, random_forest_metrics = train_and_evaluate_model(
    model=random_forest,
    model_name="random_forest_balanced",
    preprocessor=baseline_preprocessor,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    params={
        "n_estimators": 200,
        "class_weight": "balanced"
    }
)

random_forest_metrics

2026/05/21 12:58:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


accuracy: 0.8719
precision: 0.5000
recall: 0.0017
f1_score: 0.0034
roc_auc: 0.5808


,accuracy,precision,recall,f1_score,roc_auc
0,0.871856,0.5,0.001692,0.003373,0.580831


#### Observations

- Despite using class weighting, the Random Forest model failed to effectively identify lapse cases.
- Recall remained extremely low, indicating that the model predicted very few positive lapse cases.
- High accuracy is misleading in this imbalanced classification setting, as the model primarily predicted the majority class.
- ROC-AUC indicates limited discriminatory performance for lapse prediction.
- The results suggest that additional imbalance handling strategies or threshold optimization may be necessary for tree based models.

## XGBoost Classifier

XGBoost is evaluated as a gradient boosting model designed to improve predictive performance through sequential learning and error correction.

The model is capable of capturing complex nonlinear relationships and feature interactions, making it well suited for structured tabular datasets such as insurance policy data.

XGBoost also provides built in regularization and flexible handling of imbalanced classification problems, which may improve lapse prediction performance compared to baseline linear and ensemble bagging models.

In [13]:
xgboost_classifier = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

xgboost_pipeline, xgboost_metrics = train_and_evaluate_model(
    model=xgboost_classifier,
    model_name="xgboost_baseline",
    preprocessor=baseline_preprocessor,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    params={
        "n_estimators": 300,
        "learning_rate": 0.05,
        "max_depth": 5,
        "subsample": 0.8,
        "colsample_bytree": 0.8
    }
)

xgboost_metrics

2026/05/21 12:59:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


accuracy: 0.8708
precision: 0.1429
recall: 0.0017
f1_score: 0.0033
roc_auc: 0.5984


,accuracy,precision,recall,f1_score,roc_auc
0,0.870772,0.142857,0.001692,0.003344,0.598365


#### Observations

- XGBoost showed very limited ability to identify lapse cases in its baseline configuration.
- Recall remained extremely low, indicating that the model predicted very few positive lapse cases.
- High accuracy is misleading in this imbalanced classification setting due to dominant majority class predictions.
- ROC-AUC was slightly better than Random Forest but still indicates only moderate discriminatory performance.
- The results suggest that additional imbalance handling techniques or threshold optimization may be necessary to improve lapse detection performance.

## CatBoost Classifier

CatBoost is evaluated as a gradient boosting algorithm specifically designed for efficient handling of categorical features.

Unlike traditional boosting models, CatBoost applies ordered boosting and advanced categorical encoding techniques internally, which can help reduce overfitting and improve performance on structured tabular datasets.

The model is evaluated to determine whether native categorical feature handling can improve policy lapse prediction performance compared to previous baseline models.

In [14]:
catboost_classifier = CatBoostClassifier(
    iterations=300,
    learning_rate=0.05,
    depth=6,
    random_seed=42,
    verbose=0
)

catboost_pipeline, catboost_metrics = train_and_evaluate_model(
    model=catboost_classifier,
    model_name="catboost_baseline",
    preprocessor=baseline_preprocessor,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    params={
        "iterations": 300,
        "learning_rate": 0.05,
        "depth": 6
    }
)

catboost_metrics

2026/05/21 12:59:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


accuracy: 0.8712
precision: 0.2000
recall: 0.0017
f1_score: 0.0034
roc_auc: 0.6074


,accuracy,precision,recall,f1_score,roc_auc
0,0.871206,0.2,0.001692,0.003356,0.607391


#### Observations

- CatBoost achieved the highest ROC-AUC among the evaluated baseline models, indicating slightly improved ranking performance.
- Despite the ROC-AUC improvement, recall remained extremely low, showing that the model still failed to effectively identify lapse cases.
- High accuracy is misleading in this imbalanced classification setting due to dominant majority class predictions.
- The consistently low recall across all evaluated models suggests that class imbalance remains a major challenge and is not sufficiently addressed through baseline model selection alone.

## Model Comparison

| Model | Accuracy | Precision | Recall | F1-Score | ROC-AUC |
|---|---:|---:|---:|---:|---:|
| Logistic Regression (Balanced) | 0.5917 | 0.1721 | 0.5736 | 0.2647 | 0.6068 |
| Random Forest (Balanced) | 0.8719 | 0.5000 | 0.0017 | 0.0034 | 0.5808 |
| XGBoost Baseline | 0.8708 | 0.1429 | 0.0017 | 0.0033 | 0.5984 |
| CatBoost Baseline | 0.8712 | 0.2000 | 0.0017 | 0.0034 | 0.6074 |

### Overall Findings

The dataset presents a challenging imbalanced classification problem, where lapse cases represent a relatively small portion of the data.

Among the evaluated models, the balanced Logistic Regression model achieved the best overall minority class detection performance, particularly in terms of recall and F1-score, and was therefore selected as the strongest baseline model for this project.

Tree based models such as Random Forest, XGBoost, and CatBoost achieved higher overall accuracy but struggled to correctly identify lapse cases under the default classification threshold.

CatBoost achieved the highest ROC-AUC score, indicating comparatively better ranking capability, although minority class detection remained limited across all evaluated models.

## Final Model Selection
The balanced Logistic Regression model was selected as the final baseline model due to its substantially stronger minority class detection performance compared to the evaluated tree based models.

Although overall accuracy was lower, the model achieved significantly higher recall and F1-score, making it more suitable for identifying potential policy lapse cases in an imbalanced classification setting.

## Model Interpretation

To better understand the behavior of the selected Logistic Regression model, feature coefficients are analyzed to identify the variables most strongly associated with policy lapse prediction.

Positive coefficients increase the likelihood of predicting a policy lapse, while negative coefficients decrease the predicted lapse probability.

Analyzing model coefficients provides additional business insight into the factors influencing customer renewal behavior and policy retention.

In [15]:
feature_names = (
    logreg_pipeline.named_steps["preprocessor"]
    .get_feature_names_out()
)

coefficients = (
    logreg_pipeline.named_steps["model"]
    .coef_[0]
)

coefficient_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients
})

coefficient_df["abs_coefficient"] = (
    coefficient_df["coefficient"].abs()
)

coefficient_df = coefficient_df.sort_values(
    by="abs_coefficient",
    ascending=False
)

coefficient_df.head(15)

,feature,coefficient,abs_coefficient
4,categorical__polholder_diffdriver_commercial,-1.356626,1.356626
14,categorical__policy_caruse_commercial,0.981789,0.981789
46,categorical__vehicl_region_Reg2,-0.893812,0.893812
57,numerical__prem_final,0.623574,0.623574
16,categorical__policy_caruse_unknown,-0.595583,0.595583
5,categorical__polholder_diffdriver_learner 17,0.553699,0.553699
8,categorical__polholder_diffdriver_unknown,0.502998,0.502998
33,categorical__vehicl_powerkw_200 kW,-0.468375,0.468375
43,categorical__vehicl_region_Reg12,0.440902,0.440902
15,categorical__policy_caruse_private or freelanc...,-0.407926,0.407926


### Key Model Insights

- Commercial vehicle usage showed one of the strongest positive associations with policy lapse prediction.
- Higher final premium values increased the predicted likelihood of lapse, supporting earlier EDA findings related to price sensitivity.
- Certain driver profiles, such as learner and unknown driver categories, were associated with higher lapse risk.
- Bonus Malus evolution also influenced predictions, with decreasing bonus status associated with higher lapse likelihood.
- Some regional and vehicle related categories demonstrated meaningful relationships with lapse behavior, indicating the importance of geographical and vehicle characteristics in customer retention patterns.

#### Observations

- Positive coefficients increase the predicted probability of policy lapse, while negative coefficients reduce it.
- Premium pricing, vehicle usage, driver profile, and regional characteristics emerged as important predictive factors in the Logistic Regression model.
- The model coefficients are broadly consistent with patterns observed during exploratory data analysis.
- Logistic Regression provides useful interpretability for understanding the drivers of policy lapse behavior.

## Final Conclusion

This project explored multiple machine learning approaches for predicting vehicle insurance policy lapse behavior using customer, policy, premium, and vehicle related features.

Exploratory data analysis identified several meaningful relationships between policy lapse behavior and variables such as premium amount, policy duration, vehicle usage, and driver characteristics.

Among the evaluated baseline models, the balanced Logistic Regression model achieved the strongest minority class detection performance, particularly in terms of recall and F1-score, making it the most suitable baseline model for identifying potential lapse cases in the imbalanced dataset.

Tree based ensemble models achieved higher overall accuracy but struggled to correctly identify lapse cases under the default classification threshold.

The results highlight the challenges of imbalanced classification in insurance lapse prediction and suggest that further improvements may require advanced imbalance-handling techniques, threshold optimization, feature engineering, or hyperparameter tuning.


## Model Serialization

In [16]:
os.makedirs("../artifacts/model", exist_ok=True)

joblib.dump(
    logistic_regression,
    "../artifacts/model/modelv2.pkl"
)

joblib.dump(
    baseline_preprocessor,
    "../artifacts/model/preprocessor.pkl"
)

print("Artifacts saved successfully.")

Artifacts saved successfully.


---